In [ ]:
from nilearn import image, plotting
import numpy as np

# Load atlas
aal_img = image.load_img(r"C:\Users\paulo\Desktop\AAL3v2_for_SPM12\AAL3\ROI_MNI_V7.nii")
aal_data = aal_img.get_fdata()

In [ ]:
import xml.etree.ElementTree as ET

def parse_aal3_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    labels = {}

    for label in root.findall(".//label"):
        idx = int(label.find("index").text)
        name = label.find("name").text.strip()
        labels[idx] = name

    return labels


aal_labels = parse_aal3_xml(
    r"C:\Users\paulo\Desktop\AAL3v2_for_SPM12\AAL3\ROI_MNI_V7.xml"
)

print(list(aal_labels.items())[:10])

In [ ]:
# --- REGION MASKS ---

# Inferior frontal gyrus (opercular)
ifg = ((aal_data == 7) | (aal_data == 8))

# Medial frontal cortex
mfc = ((aal_data == 19) | (aal_data == 20))

# Caudate
caudate = ((aal_data == 75) | (aal_data == 76))

# Hippocampus
hipp = ((aal_data == 41) | (aal_data == 42))

# Posterior cingulate
pcc = ((aal_data == 39) | (aal_data == 40))

# Precuneus
precuneus = ((aal_data == 71) | (aal_data == 72))

# Amygdala
amygdala = ((aal_data == 45) | (aal_data == 46))

# Gyrus rectus / medial orbitofrontal
rectus = ((aal_data == 23) | (aal_data == 24))

# Putamen
putamen = ((aal_data == 77) | (aal_data == 78))

# Pallidum
pallidum = ((aal_data == 79) | (aal_data == 80))

# Superior parietal lobule (sensorimotor integration)
spl = ((aal_data == 63) | (aal_data == 64))

# Postcentral gyrus (proprioception)
postcentral = ((aal_data == 61) | (aal_data == 62))

# Cerebellum
cerebellum = (
    (aal_data == 103) | (aal_data == 104) |  # Lobule VI
    (aal_data == 105) | (aal_data == 106)    # Lobule VIIb
)

# # Thalamus
thalamus = ((aal_data >= 121) & (aal_data <= 150))

# Insula
insula = ((aal_data == 33) | (aal_data == 34))

In [ ]:
combined = np.zeros_like(aal_data)

# --- Executive / frontal ---
combined[ifg] = 1
combined[mfc] = 2
combined[rectus] = 3

# --- Limbic / navigation ---
combined[hipp] = 4
combined[amygdala] = 5
combined[pcc] = 6
combined[precuneus] = 7

# --- Basal ganglia ---
combined[caudate] = 8
combined[putamen] = 9
combined[pallidum] = 10
combined[thalamus] = 11

# --- Sensorimotor ---
combined[postcentral] = 12
combined[spl] = 13
combined[insula] = 14

# --- Cerebellum ---
combined[cerebellum] = 15

combined_img = image.new_img_like(aal_img, combined)

In [ ]:
region_labels = {
    1: "Inferior frontal gyrus (opercular)",
    2: "Medial frontal cortex",
    3: "Gyrus rectus",
    4: "Hippocampus",
    5: "Amygdala",
    6: "Posterior cingulate",
    7: "Precuneus",
    8: "Caudate",
    9: "Putamen",
    10: "Pallidum",
    11: "Thalamus",
    12: "Postcentral gyrus",
    13: "Superior parietal lobule",
    14: "Insula",
    15: "Cerebellum"
}

for k, v in region_labels.items():
    print(f"{k}: {v}")

In [ ]:
from nilearn.datasets import load_mni152_template

mni = load_mni152_template()

plotting.plot_roi(
    combined_img,
    bg_img=mni,
    cmap="tab20",     # distinct colors per region
    #opacity=0.7,
    black_bg=True,
    vmin=0,
    vmax=15,
    title="FOG Network (AAL3)"
)

In [ ]:
plotting.view_img(
    combined_img,
    bg_img=mni,
    cmap="tab20",     # distinct colors per region
    opacity=0.7,
    black_bg=True,
    vmin=0,
    vmax=15,
    colorbar=True,
    title="FOG Network (AAL3)"
)